# 04 · A globe for context, 3D columns for comparison

**40 minutes · Intermediate.** Rotate a real geographic globe with fictional hazard points, understand spherical coordinates, and compare a 3D thematic map with a simpler bar chart.

[Run this lab in JupyterLite](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/lab/index.html?path=04_3D_Globe.ipynb) · [Book home](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/book/intro.html)

**Start:** run cells from top to bottom (Shift+Enter), or use Run → Run All Cells. The first launch downloads the Python runtime and packages. Core analysis uses bundled training data; internet is needed for initial setup and interactive JavaScript. Each lab starts independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "folium==0.20.0", "plotly==6.3.0"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from fieldguide import setup_style, load_tables, summarize, TRAINING
setup_style()
print(TRAINING)

## 3D is a choice of representation
A globe helps with planetary context and the date line. It also hides the far hemisphere. The map below uses bundled Natural Earth land outlines on a unit sphere; no map API key or imagery tiles are needed. Its points are **invented earthquake-format events**. The globe is not a terrain model, and point height does not encode earthquake depth.
Plotly JavaScript loads from its CDN and requires WebGL. If unavailable, use the 2D companion immediately below.

In [ ]:
from worldmonitor import earthquake_feed
from fieldguide import earthquake_table, globe, show_plotly, land_rings
data, provenance = await earthquake_feed(live=False)
events = earthquake_table(data)
print(provenance["mode"])
show_plotly(globe(events,provenance["mode"]))

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
for ring in land_rings():
    ax.fill(ring[:,0],ring[:,1],color="#dce6eb",edgecolor="#829aa8",linewidth=.3)
scatter=ax.scatter(events.longitude,events.latitude,c=events.magnitude,cmap="YlOrRd",s=65,edgecolor="#132c46")
fig.colorbar(scatter,ax=ax,label="Magnitude (synthetic)")
ax.set(xlim=(-180,180),ylim=(-60,85),xlabel="Longitude (°)",ylabel="Latitude (°)",title="SYNTHETIC events · 2D companion · Natural Earth land")
plt.show()
display(events)

## Build the sphere yourself
For longitude λ and latitude φ in radians: `x=cos(φ)cos(λ)`, `y=cos(φ)sin(λ)`, `z=sin(φ)`. A unit-radius check catches coordinate errors. Longitude spacing on a flat equirectangular map does not represent equal ground distances at all latitudes.

In [ ]:
from fieldguide import xyz
x,y,z=xyz(events.longitude.to_numpy(),events.latitude.to_numpy())
assert np.allclose(x*x+y*y+z*z,1)
print("Every point lies on the unit sphere.")

## Extrude a district measure, not terrain
The height below is reported cases per 100,000 for seven days. The horizontal axes are longitude and latitude; the vertical axis is a health-reporting measure, **not meters**. Occlusion and perspective complicate comparison, so a sorted bar chart accompanies the 3D map.

In [ ]:
s=summarize()
fig=plt.figure(figsize=(13,5))
ax=fig.add_subplot(121,projection="3d")
ax.bar3d(s.longitude-.13,s.latitude-.13,np.zeros(len(s)),.26,.26,s.reported_per_100k,color="#007f86",shade=True)
ax.set(xlabel="Longitude (°)",ylabel="Latitude (°)",zlabel="Reports / 100,000",title="SYNTHETIC thematic columns")
ax.view_init(elev=28,azim=-55)
ax2=fig.add_subplot(122)
ordered=s.sort_values("reported_per_100k")
ax2.barh(ordered.district,ordered.reported_per_100k)
ax2.set(xlabel="Reports per 100,000 · 7 days",title="Same values, easier comparison")
plt.tight_layout(); plt.show()

## Practice and debrief

Rotate the globe until an event disappears behind it. Explain how a screenshot could omit evidence. Change the 3D column camera angle and identify whether the top district remains obvious. Choose the better display for a printed sitrep and justify it.

## Sources and further learning

[Natural Earth public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/) · [Plotly 3D scatter](https://plotly.com/python/3d-scatter-plots/) · [World Monitor dual map engines](https://github.com/koala73/worldmonitor).